# Fine-tuning to follow instructions

In [1]:
import json
import requests

from typing import Optional, Any, Iterable, Dict, Tuple

import torch
from torch.utils.data import Dataset

from gptlight.models  import GPTModel
from gptlight.tokenizer import GPTTokenizer
from gptlight.utils import download_and_load_gpt2

2025-12-28 13:16:13.595836: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = GPTTokenizer()

Previously, we implemented the LLM architecture, carried out pretraining, and
imported pretrained weights from external sources into our model. Then, we
focused on fine-tuning our LLM for a specific classification task: distinguishing
between spam and non-spam text messages. Now we’ll implement the process for
fine-tuning an LLM to follow human instructions.

Instruction fine-tuning is one of the main techniques behind developing LLMs for chatbot applications, personal assistants, and other conversational tasks.

## Introduction to instruction fine-tuning

We now know that pretraining an LLM involves a training procedure where it learns to generate one word at a time. The resulting pretrained LLM is capable of text completion, meaning it can finish sentences or write text paragraphs given a fragment as input. However, pretrained LLMs often struggle with specific instructions, such as “Fix the grammar in this text” or “Convert this text into passive voice.” Later, we will examine a concrete example where we load the pretrained LLM as the basis for instruction fine-tuning, also known as supervised instruction fine-tuning.

Here, we focus on improving the LLM’s ability to follow such instructions and generate a desired response.

## Preparing a dataset for supervised instruction fine-tuning

Let’s download and format the instruction dataset for instruction fine-tuning a pre-trained LLM. The dataset consists of 1,100 instruction–response pairs.

In [3]:
def fetch_instructions_data(to_file_path: Optional[str] = None) -> Any:
    file_id = "13uAQeUAEe4y5X1MvqX57LPDSx_w_Vi1W"
    url = f"https://drive.google.com/uc?export=download&id={file_id}"

    resp = requests.get(url)
    resp.raise_for_status()

    data = resp.json()

    if to_file_path:
        with open(to_file_path, "w") as f:
            json.dump(data, f, indent=2)

    return data

In [4]:
data = fetch_instructions_data("../assets/datasets/instruction-data.json")

In [6]:
print("Number of entries:", len(data))
print("\nExample entry: \n", data[50])

Number of entries: 1100

Example entry: 
 {'instruction': 'Identify the correct spelling of the following word.', 'input': 'Ocassion', 'output': "The correct spelling is 'Occasion.'"}


Instruction fine-tuning involves training a model on a dataset where the input-output pairs, like those we extracted from the JSON file, are explicitly provided. There are
various methods to format these entries for LLMs.

The following diagrams illustrate two different example formats, often referred to as *prompt styles*, used in the training of notable
LLMs such as Alpaca and Phi-3.

+-------------------------------------------------------------------------+
| {                                                                       |
|   "instruction": "Identify the correct spelling of the following word.",|
|   "input": "Ocassion",                                                  |
|   "output": "The correct spelling is 'Occasion.'"                       |
| }                                                                       |
+-------------------------------------------------------------------------+
                    |
                    | Apply Alpaca prompt style template.
                    v
+------------------------------------------------------------------------+
| Below is an instruction that describes a task. Write a response that   |
| appropriately completes the request.                                   |
|                                                                        |
| ### Instruction:                                                       |
| Identify the correct spelling of the following word.                   |
|                                                                        |
| ### Input:                                                             |
| Ocassion                                                               |
|                                                                        |
| ### Response:                                                          |
| The correct spelling is 'Occasion'.                                    |
+------------------------------------------------------------------------+

+-------------------------------------------------------------------------+
| {                                                                       |
|   "instruction": "Identify the correct spelling of the following word.",|
|   "input": "Ocassion",                                                  |
|   "output": "The correct spelling is 'Occasion.'"                       |
| }                                                                       |
+-------------------------------------------------------------------------+
                                |                                         
                                | Apply Phi-3 prompt style template.
                                v
+-------------------------------------------------------------------------+
| <|user|>                                                                |
| Identify the correct spelling of the following word: 'Ocassion'         |
|                                                                         |
| <|assistant|>                                                           |
| The correct spelling is 'Occasion'.                                     |
+-------------------------------------------------------------------------+

Alpaca was one of the early LLMs to publicly detail its instruction fine-tuning process. Phi-3, developed by Microsoft, is included to demonstrate the diversity in prompt styles. In the rest of this notebook, we use the Alpaca prompt style since it is one of the most popular ones, largely because it helped define the original approach to fine-tuning.

Let’s define a `format_input` function that we can use to convert the entries in the data list into the Alpaca-style input format.

In [7]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    input_text = (
        f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    )
    return instruction_text + input_text

This format_input function takes a dictionary entry as input and constructs a formatted string. Let’s test it to dataset entry data[50], which we looked at earlier :

In [8]:
model_input = format_input(data[50])
desired_response = f"\n\n### Response:\n{data[50]['output']}"
print(model_input + desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Identify the correct spelling of the following word.

### Input:
Ocassion

### Response:
The correct spelling is 'Occasion.'


In [9]:
model_input = format_input(data[999])
desired_response = f"\n\n### Response:\n{data[999]['output']}"
print(model_input + desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What is an antonym of 'complicated'?

### Response:
An antonym of 'complicated' is 'simple'.


Before we move on to setting up the PyTorch data loaders in the next section, let’s divide the dataset into training, validation, and test sets analogous to what we have done with the spam classification dataset.

In [10]:
n = len(data)
train_size = int(n*0.85)
test_size = int(n*0.1)
val_size = n - train_size - test_size

train_data = data[:train_size]
test_data = data[train_size:train_size+test_size]
val_data = data[train_size+test_size:]

print("Training set length:", len(train_data))
print("Validation set length:", len(val_data))
print("Test set length:", len(test_data))

Training set length: 935
Validation set length: 55
Test set length: 110


## Organizing data into training batches

In the previous sections, the training batches were created automatically by the PyTorch DataLoader class, which employs a default collate function to combine lists of samples into batches. A collate function is responsible for taking a list of individual data samples and merging them into a single batch that can be processed efficiently by the model during training.

However, the batching process for instruction fine-tuning is a bit more involved and requires us to create our own custom collate function that we will later plug into the DataLoader. We implement this custom collate function to handle the specific requirements and formatting of our instruction fine-tuning dataset.

> Implementing an instruction dataset class

In [14]:
class InstructionDataset(Dataset):
    
    def __init__(self, 
        data:Iterable[Dict[str, Any]], 
        tokenizer:GPTTokenizer
    ):
        self.data = data
        self.encoded_texts = []
        for entry in data:
            instruction_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_input + response_text
            self.encoded_texts.append(
                tokenizer.encode(full_text).tolist()
            )
        
    def __getitem__(self, index):
        return self.encoded_texts[index]
    
    def __len__(self):
        return len(self.data)

Similar to the approach used for classification fine-tuning, we want to accelerate training by collecting multiple training examples in a batch, which necessitates padding all inputs to a similar length. As with classification fine-tuning, we use the <|endoftext|> token as a padding token.

Instead of appending the <|endoftext|> tokens to the text inputs, we can append
the token ID corresponding to <|endoftext|> to the pretokenized inputs directly.

> We can implement the padding process with a custom collate function:

In [15]:
def custom_collate_draft_1(
    batch:Iterable,
    device:torch.device,
    pad_token_id:int=50256
) -> torch.Tensor:
    batch_max_length = max(len(item)+1 for item in batch)
    inputs_lst = []
    
    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]
        
        padded = (
            new_item + [pad_token_id]*(batch_max_length - len(new_item))
        )
        inputs = torch.tensor(padded[:-1])
        
        inputs_lst.append(inputs)
    
    return torch.stack(inputs_lst).to(device)

The custom_collate_draft_1 we implemented is designed to be integrated into a
PyTorch DataLoader, but it can also function as a standalone tool. Here, we use it independently to test and verify that it operates as intended

In [16]:
inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]

batch = (
    inputs_1,
    inputs_2,
    inputs_3
)
print(custom_collate_draft_1(batch, device=device))

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])


We have just implemented our first custom collate function to create batches from lists of inputs. However, as we previously learned, we also need to create batches with the target token IDs corresponding to the batch of input IDs. These target IDs are crucial because they represent what we want the model to generate and what we need during training to calculate the loss for the weight updates. That is, we modify our custom collate function to return the target token IDs in addition to the input token IDs.

Similar to the process we used to pretrain an LLM, the target token IDs match the input token IDs but are shifted one position to the right. This setup allows the LLM to learn how to predict the next token in a sequence.

In [17]:
def custom_collate_draft_2(
    batch:Iterable,
    device:torch.device,
    pad_token_id:int=50256
) -> Tuple[torch.Tensor]:
    batch_max_length = max(len(item)+1 for item in batch)
    inputs_lst, targets_lst = [], []
    
    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]
        
        padded = (
            new_item + [pad_token_id]*(batch_max_length - len(new_item))
        )
        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])
        inputs_lst.append(inputs)
        targets_lst.append(targets)
        
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)
    return inputs_tensor, targets_tensor

In [18]:
inputs, targets = custom_collate_draft_2(batch, device=device)
print(inputs)
print(targets)

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])
tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256, 50256, 50256, 50256],
        [    8,     9, 50256, 50256, 50256]])


In the next step, we assign a -100 placeholder value to all padding tokens. This special value allows us to exclude these padding tokens from contributing to the training loss calculation, ensuring that only meaningful data influences model learning. We will discuss this process in more detail after we implement this modification. (When fine-tuning for classification, we did not have to worry about this since we only trained the model based on the last output token.)

However, note that we retain one end-of-text token, ID 50256, in the target list. Retaining it allows the LLM to learn when to generate an end-of-text token in response to instructions, which we use as an indicator that the gener-
ated response is complete.

In the following listing, we modify our custom collate function to replace tokens with ID 50256 with -100 in the target lists. Additionally, we introduce an allowed_max_length parameter to optionally limit the length of the samples. This adjustment will be useful if you plan to work with your own datasets that exceed the 1,024-token context size supported by the GPT-2 model.

>  Implementing a custom batch collate function

In [33]:
def custom_collate_fn(
    batch:Iterable,
    device:torch.device,
    pad_token_id:int=50256,
    ignore_index:int=-100,
    allowed_max_length:Optional[int]=None
)-> Tuple[torch.Tensor]:
    
    batch_max_lenght = max(len(item)+1 for item in batch)
    inputs_lst, targets_lst = [], []
    
    for item in batch:
        
        new_item = item.copy()
        new_item += [pad_token_id]
        
        padded = (
            new_item + [pad_token_id]*(batch_max_lenght - len(new_item))
        )
        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])
        
        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        
        if indices.numel()>1:
            targets[indices[1:]] = ignore_index
        
        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]
        
        inputs_lst.append(inputs)
        targets_lst.append(targets)
    
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)
    
    return inputs_tensor, targets_tensor

Again, let’s try the collate function on the sample batch that we created earlier to check that it works as intended:

In [34]:
inputs, targets = custom_collate_fn(batch, device=device)
print(inputs)
print(targets)

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])
tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256,  -100,  -100,  -100],
        [    8,     9, 50256,  -100,  -100]])


In [2]:
settings, params = download_and_load_gpt2("355M", "../assets/gpt2")

checkpoint: 100%|██████████| 77.0/77.0 [00:00<00:00, 148kiB/s]
encoder.json: 100%|██████████| 1.04M/1.04M [00:00<00:00, 1.27MiB/s]
hparams.json: 100%|██████████| 91.0/91.0 [00:00<00:00, 162kiB/s]
model.ckpt.data-00000-of-00001: 100%|██████████| 1.42G/1.42G [20:54<00:00, 1.13MiB/s] 
model.ckpt.index: 100%|██████████| 10.4k/10.4k [00:00<00:00, 2.92MiB/s]
model.ckpt.meta: 100%|██████████| 927k/927k [00:00<00:00, 1.31MiB/s]
vocab.bpe: 100%|██████████| 456k/456k [00:02<00:00, 218kiB/s]  
2025-12-10 12:01:53.383850: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 205852672 exceeds 10% of free system memory.
